# Algorithm Forge — Phase 4 Mutator Fine-Tune

Fine-tune `Qwen2.5-Coder-7B` (or Gemma 4) on the mutation log produced by your Algorithm Forge runs, then export a GGUF you can load into Ollama as the `ollama-forge-mutator` provider.

## Before you start

1. **Runtime → Change runtime type → T4 GPU** (free Colab tier is enough).
2. Build and upload your dataset first:
   ```bash
   uv run science-game build-mutator-dataset --runs-root runs \
       --out datasets/mutator-v1.jsonl --mode sft --min-delta 0.001
   hf upload-dataset Beko2210/algorithm-forge-mutations-v1 datasets/mutator-v1.jsonl
   ```
3. You'll need an HF write token (`HF_TOKEN`) for the model push step.

In [ ]:
# Install Unsloth (auto-detects Colab GPU)
!pip install -qU unsloth
!pip install -qU "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

In [ ]:
# --- Configuration ---
BASE_MODEL = "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit"
DATASET_REPO = "Beko2210/algorithm-forge-mutations-v1"
OUT_MODEL_REPO = "Beko2210/algorithm-forge-mutator-qwen-v1"

MAX_SEQ_LEN = 4096
LORA_R = 16
LORA_ALPHA = 16
LR = 2e-4
EPOCHS = 2
BATCH_SIZE = 2
GRAD_ACCUM = 4

In [ ]:
# --- Load base model (4-bit) ---
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

In [ ]:
# --- Load the mutation dataset from HF Hub ---
from datasets import load_dataset
from unsloth.chat_templates import standardize_sharegpt

ds = load_dataset(DATASET_REPO, split="train")
ds = standardize_sharegpt(ds)
print(ds)
print(ds[0])

In [ ]:
# --- Train ---
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    args=TrainingArguments(
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=10,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        logging_steps=10,
        optim="adamw_8bit",
        seed=42,
        output_dir="forge-mutator-out",
    ),
)
trainer.train()

In [ ]:
# --- Quick sanity check ---
FastLanguageModel.for_inference(model)
test_prompt = (
    "Below is the current best program. Produce an improved variant.\n\n"
    "```python\n"
    "def matmul2x2(A, B, mul):\n"
    "    a, b = A[0][0], A[0][1]\n"
    "    c, d = A[1][0], A[1][1]\n"
    "    e, f = B[0][0], B[0][1]\n"
    "    g, h = B[1][0], B[1][1]\n"
    "    return [[mul(a,e)+mul(b,g), mul(a,f)+mul(b,h)],\n"
    "            [mul(c,e)+mul(d,g), mul(c,f)+mul(d,h)]]\n"
    "```\n"
)
inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
out = model.generate(**inputs, max_new_tokens=512, temperature=0.8, do_sample=True)
print(tokenizer.decode(out[0]))

In [ ]:
# --- Export GGUF (Q4_K_M = fast; Q8_0 = higher fidelity) ---
model.save_pretrained_gguf("forge-mutator-q4km", tokenizer, quantization_method="q4_k_m")
# model.save_pretrained_gguf("forge-mutator-q8", tokenizer, quantization_method="q8_0")

In [ ]:
# --- Push to Hugging Face Hub ---
from huggingface_hub import login
login()  # paste HF token when prompted

model.push_to_hub_gguf(OUT_MODEL_REPO, tokenizer, quantization_method="q4_k_m")

## After Colab

Back on your local PC:

```bash
# Download the GGUF
hf download Beko2210/algorithm-forge-mutator-qwen-v1 \
    --include '*.gguf' --local-dir ./phase4-out

# Register with Ollama
uv run science-game phase4 prepare \
    --dataset datasets/mutator-v1.jsonl \
    --gguf-name model-Q4_K_M.gguf

cd phase4-out
ollama create algorithm-forge-mutator -f Modelfile

# A/B compare against base Qwen
uv run science-game phase4 evaluate \
    --benchmark matmul \
    --base-provider ollama-qwen \
    --finetuned-provider ollama-qwen \
    --finetuned-model algorithm-forge-mutator \
    --seeds 0,1,2,3,4 --generations 20
```

The report (`phase4-out/ab-report.json`) tells you whether the fine-tuned mutator actually outperformed the base — if yes, you've closed the self-improvement loop.